# A3-M1 · 06 임베딩으로 유사도
e5(모델 다운로드)와 OpenAI 임베딩(.env의 OPENAI_API_KEY)으로 같은 평가를 합니다.

## 14-1. Hugging Face 임베딩 — e5로 문장 벡터 만들기 (모델 다운로드 필요)

In [1]:
import torch, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from shop_faq import FAQ, TESTS

name = "intfloat/multilingual-e5-small"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModel.from_pretrained(name).eval()

@torch.inference_mode()
def e5(texts, kind):
    b = tok([f"{kind}: {t}" for t in texts], padding=True, truncation=True, max_length=256, return_tensors="pt")
    h = model(**b).last_hidden_state
    m = b["attention_mask"].unsqueeze(-1).float()
    return F.normalize((h * m).sum(1) / m.sum(1), dim=1)

D = e5([f"{q} {a}" for _, _, q, a in FAQ], "passage")
print("문서 벡터:", tuple(D.shape))
ids = [i for i, *_ in FAQ]
hit1 = 0
for q, gold in TESTS:
    s = D @ e5([q], "query")[0]
    top = s.argsort(descending=True)[:3].tolist()
    hit1 += ids[top[0]] == gold
    print(f"{q:<24} → {[ids[i] for i in top]}  {[round(float(s[i]), 3) for i in top]}")
print("e5 Hit@1 =", hit1, "/", len(TESTS))

c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2227.68it/s]


문서 벡터: (20, 384)
택배 언제 와요?                → [101, 301, 103]  [0.846, 0.827, 0.824]
배송비 무료 기준이 뭐예요?          → [102, 304, 105]  [0.904, 0.863, 0.846]
내 택배 지금 어디쯤인지 보고 싶어요     → [103, 105, 502]  [0.874, 0.85, 0.843]
소파 같은 큰 물건은 어떻게 와요?      → [104, 303, 304]  [0.821, 0.806, 0.796]
미국으로 보내줄 수 있나요?          → [105, 502, 301]  [0.866, 0.849, 0.844]
카드 할부 몇 개월까지 돼요?         → [202, 401, 301]  [0.901, 0.858, 0.85]
받은 지 5일 됐는데 반품 돼요?       → [301, 401, 101]  [0.892, 0.872, 0.87]
물건이 깨져서 왔어요              → [303, 301, 602]  [0.86, 0.833, 0.827]
환불 며칠 걸려요?               → [401, 402, 301]  [0.885, 0.871, 0.87]
주문 취소 어떻게 해요?            → [501, 401, 301]  [0.884, 0.858, 0.837]
비번 까먹었어요                 → [303, 501, 601]  [0.823, 0.81, 0.809]
포인트 유효기간 있나요?            → [701, 301, 401]  [0.846, 0.841, 0.829]
e5 Hit@1 = 11 / 12


## 14-2. OpenAI 임베딩 — 보내기 전에 tiktoken으로 세기 (API 키 필요)

In [ ]:
import os, numpy as np, tiktoken
from dotenv import load_dotenv
from openai import OpenAI
from shop_faq import FAQ, TESTS
load_dotenv(".env")

enc = tiktoken.encoding_for_model("text-embedding-3-small")
texts = [f"{q} {a}" for _, _, q, a in FAQ]
n_tok = [len(enc.encode(t)) for t in texts]
print("문서 토큰 수 합계:", sum(n_tok), "| 가장 긴 문서:", max(n_tok), "토큰 (한 입력 최대 8,192)")

client = OpenAI()
def oa(texts, dim=None):
    kw = {"dimensions": dim} if dim else {}
    r = client.embeddings.create(model="text-embedding-3-small", input=texts, **kw)
    return np.array([d.embedding for d in r.data], dtype="float32")

D = oa(texts)
print("문서 벡터:", D.shape, "| 길이:", np.linalg.norm(D[0]).round(3))
ids = [i for i, *_ in FAQ]
hit1 = 0
for q, gold in TESTS:
    s = D @ oa([q])[0]
    top = s.argsort()[::-1][:3]
    hit1 += ids[top[0]] == gold
    print(f"{q:<24} → {[ids[i] for i in top]}  {[round(float(x), 3) for x in s[top]]}")
print("OpenAI Hit@1 =", hit1, "/", len(TESTS))